# Notebook 4 — Stage B: Small CNN
**Spurious Feature Detection before OOD failure — Person 4 (CNN)**

How to run on Kaggle:
1. Settings → Accelerator **GPU T4 x1** (CPU also works, just slower), Internet **on** (MNIST download).
2. Run the **smoke test** cell interactively first. If it prints `CNN SMOKE TEST PASSED`, click **Save Version → Save & Run All (Commit)**.
3. The committed version is the freeze. Everything is written to `/kaggle/working/runs_cnn/`.

Rules (from the plan): do **not** edit the shared-core section; hyperparameters were fixed on E1 validation only; OOD evaluation runs only after `freeze_scores()`.

## Part 1 — Shared core (copied verbatim from the updated Notebook 0 (design-based registry + MI + permutation importance) — DO NOT EDIT)
If the team later adds `%%writefile core.py` to Notebook 0, replace this whole section with `Add Input → Notebook 0 output` and `from core import *`.

In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Imports successful!")
print("PyTorch version:", torch.__version__)


Imports successful!
PyTorch version: 2.11.0+cu130


In [2]:

mnist_train = datasets.MNIST(
    root="/kaggle/working/data",
    train=True,
    download=True
)

mnist_test = datasets.MNIST(
    root="/kaggle/working/data",
    train=False,
    download=True
)

# Convert images and labels into NumPy arrays
train_x_all = mnist_train.data.numpy()
train_y_all = mnist_train.targets.numpy()

test_x = mnist_test.data.numpy()
test_y = mnist_test.targets.numpy()

# Binary labels: even = 0, odd = 1
train_y_all = (train_y_all % 2).astype(np.int64)
test_y = (test_y % 2).astype(np.int64)

# Split training data into train and validation sets
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(train_x_all))

split = int(0.8 * len(indices))
train_idx = indices[:split]
val_idx = indices[split:]

train_x = train_x_all[train_idx]
train_y = train_y_all[train_idx]

val_x = train_x_all[val_idx]
val_y = train_y_all[val_idx]

print("Training examples:", len(train_x))
print("Validation examples:", len(val_x))
print("Test examples:", len(test_x))
print("Binary classes:", np.unique(train_y))


100%|██████████| 9.91M/9.91M [00:00<00:00, 36.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.18MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.2MB/s]

Training examples: 48000
Validation examples: 12000
Test examples: 10000
Binary classes: [0 1]


In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, mutual_info_score
from itertools import combinations
from math import factorial
import pandas as pd

ATTRS = ["color", "marker_tl", "marker_br"]
PATCH = {"marker_tl": (slice(1, 4), slice(1, 4)),     # top-left 3x3
         "marker_br": (slice(24, 27), slice(24, 27))} # bottom-right 3x3

def make_multi_attr_mnist(images, labels, corrs, seed=0):
    """corrs: dict attr -> label correlation (0 = independent decoy)."""
    rng = np.random.default_rng(seed)
    gray = images.astype(np.float32) / 255.0
    n = len(labels)
    out = np.zeros((n, 3, 28, 28), dtype=np.float32)
    attrs = {}
    for a in ATTRS:
        c = corrs.get(a, 0.0)
        match = rng.random(n) < (1 + c) / 2
        attrs[a] = np.where(match, labels, 1 - labels).astype(np.int64)
    out[attrs["color"] == 0, 0] = gray[attrs["color"] == 0]   # red
    out[attrs["color"] == 1, 1] = gray[attrs["color"] == 1]   # green
    for a, (r, c) in PATCH.items():                           # markers: blue channel only
        out[attrs[a] == 1, 2, r, c] = 1.0
    return out, attrs


In [4]:
def intervene(images, attr):
    x = images.copy()
    if attr == "color":
        x[:, [0, 1]] = images[:, [1, 0]]                 # swap red/green
    elif attr in PATCH:
        r, c = PATCH[attr]
        x[:, 2, r, c] = 1.0 - images[:, 2, r, c]         # toggle marker on/off
    elif attr == "null":
        pass                                             # do-nothing control
    else:
        raise ValueError(attr)
    return x

In [5]:
def build_registry(labels, attrs, threshold=0.5, design_corrs=None):
    """
    Register spuriousness from the experiment design, not noisy sample correlation.
    Convention: strictly greater than threshold is spurious; exactly threshold is not.
    Observed sample correlations are retained as diagnostics.
    """
    reg = {}
    for a, v in attrs.items():
        observed = float(np.corrcoef(labels, v)[0, 1])
        designed = float(design_corrs[a]) if design_corrs is not None and a in design_corrs else observed
        reg[a] = {
            "train_corr": observed,
            "design_corr": designed,
            "spurious": abs(designed) > threshold
        }
    return reg

flat = lambda x: x.reshape(len(x), -1)


In [6]:
def sensitivity_score(model, X, attr):
    """Main score (Person 2's job): fraction of predictions that flip."""
    p0 = model.predict(flat(X))
    p1 = model.predict(flat(intervene(X, attr)))
    return float(np.mean(p0 != p1))

def label_corr_score(labels, attrs, attr):
    """Data-only baseline."""
    return abs(float(np.corrcoef(labels, attrs[attr])[0, 1]))

def mutual_information_score(labels, attrs, attr):
    """Data-only baseline: mutual information between training labels and a
    training attribute (discrete, computed on E1 training data)."""
    return float(mutual_info_score(np.asarray(labels).astype(int),
                                   np.asarray(attrs[attr]).astype(int)))

def grouped_attribute_permutation_score(model, X, y, attr_values, attr,
                                        n_repeats=5, seed=12345):
    """Grouped permutation importance on held-out validation data.
    Shuffle one attribute across examples, rebuild that attribute in the images,
    and measure the accuracy drop.  Score = baseline acc - mean permuted acc.
    (Different from intervene(), which toggles the attribute within each example.)"""
    X = np.asarray(X); y = np.asarray(y)
    current = np.asarray(attr_values).astype(np.int64)
    if not (len(X) == len(y) == len(current)):
        raise ValueError("X, y and attr_values must have equal lengths.")
    if attr != "color" and attr not in PATCH:
        raise ValueError(f"Unknown attribute: {attr}")
    baseline_acc = accuracy_score(y, model.predict(flat(X)))
    rng = np.random.default_rng(seed)
    drops = []
    for _ in range(n_repeats):
        shuffled = rng.permutation(current)
        X_perm = X.copy()
        if attr == "color":
            changed = shuffled != current            # swap R/G only where assigned colour changed
            X_perm[changed, 0] = X[changed, 1]
            X_perm[changed, 1] = X[changed, 0]
        else:
            r, c = PATCH[attr]                       # rebuild marker from shuffled values
            X_perm[:, 2, r, c] = shuffled[:, None, None].astype(X_perm.dtype)
        drops.append(baseline_acc - accuracy_score(y, model.predict(flat(X_perm))))
    return float(np.mean(drops))


In [7]:
def stability_score(model, X, y, attrs, attr):
    pred = model.predict(flat(X))
    agree = attrs[attr] == y
    if agree.sum() < 20 or (~agree).sum() < 20:
        return np.nan                     # too few samples in one group
    return float(accuracy_score(y[agree], pred[agree])
                 - accuracy_score(y[~agree], pred[~agree]))

In [8]:
def attribute_shapley(model, X, y, attr_list):
    k = len(attr_list)
    def value(S):
        x = X
        for a in S:
            x = intervene(x, a)
        proba = model.predict_proba(flat(x))
        return proba[np.arange(len(y)), y]          # P(true class)
    cache = {frozenset(S): value(S)
             for r in range(k + 1) for S in combinations(attr_list, r)}
    phi = {}
    for a in attr_list:
        others = [b for b in attr_list if b != a]
        tot = np.zeros(len(y))
        for r in range(k):
            w = factorial(r) * factorial(k - r - 1) / factorial(k)
            for S in combinations(others, r):
                tot += w * (cache[frozenset(S)] - cache[frozenset(S) | {a}])
        phi[a] = float(np.mean(np.abs(tot)))
    return phi

In [9]:
def run_task(task_id, model_fn, n_score=2000):
    rng = np.random.default_rng(1000 + task_id)
    order = rng.permutation(ATTRS)
    corrs = {order[0]: 0.9, order[1]: 0.6, order[2]: 0.0}   # strong, weak, decoy

    Xtr, Atr = make_multi_attr_mnist(train_x, train_y, corrs, seed=2000 + task_id)
    Xva, Ava = make_multi_attr_mnist(val_x,   val_y,   corrs, seed=3000 + task_id)
    reg = build_registry(train_y, Atr, threshold=0.5, design_corrs=corrs)

    model = model_fn().fit(flat(Xtr), train_y)
    Xs, ys = Xva[:n_score], val_y[:n_score]
    As = {a: v[:n_score] for a, v in Ava.items()}
    shap = attribute_shapley(model, Xs, ys, ATTRS)

    rows = []
    for a in ATTRS:
        rows.append({
            "task": task_id, "attr": a, "true_corr": corrs[a],
            "spurious": reg[a]["spurious"],
            "sensitivity": sensitivity_score(model, Xs, a),
            "label_corr":  label_corr_score(train_y, Atr, a),
            "stability":   stability_score(model, Xs, ys, As, a),
            "shap":        shap[a],
        })
    null = sensitivity_score(model, Xs, "null")
    return rows, order[0], null

def evaluate(all_rows, strongest,
             methods=("sensitivity", "label_corr", "stability", "shap")):
    df = pd.DataFrame(all_rows)
    out = {}
    for m in methods:
        top1 = np.mean([df[df.task == t].sort_values(m, ascending=False).attr.iloc[0] == s
                        for t, s in strongest.items()])
        sub = df.dropna(subset=[m])
        auroc = roc_auc_score(sub.spurious, sub[m]) if sub.spurious.nunique() == 2 else np.nan
        out[m] = {"top1": top1, "auroc": auroc}
    return df, pd.DataFrame(out).T

In [10]:
#  CONFIG 
import os, json, time, hashlib, itertools, pickle, shutil, warnings, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, t as t_dist
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./working")

CONFIG = {
    "out_dir": f"{WORK}/runs_mi_perm_v2",  # fresh dir: v1 used an intervention flip, not a true permutation
    # ---- experiment grid (cut down to what the team can finish) ----
    "grid": {
        "arch":       ["lr"],                    # Stage B: add "rf", "mlp", "cnn"
        "train_corr": [0.5, 0.7, 0.85, 0.95],
        "size":       [5000, 20000],
        "seed":       [0, 1, 2, 3, 4],
        "reg":        ["default"],               # stretch: ["low", "default", "high"]
        "aug":        ["none"],                  # stretch: ["none", "color_rand"]
    },
    "shortcut":    "color",                      # attribute whose correlation is swept
    "decoy_corrs": {"marker_tl": 0.0, "marker_br": 0.0},   # label-independent decoys
    "aug_p":       0.5,                          # fraction of train images colour-swapped under "color_rand"
    # ---- OOD environments (colour-label correlation at test time) ----
    "ood_envs":    {"E2": 0.75, "E3": 0.50, "E4": 0.25, "E5": 0.00, "REV": -0.95},
    # ---- scoring ----
    "n_score":     2000,                         # held-out val samples used for scoring
    "spurious_threshold": 0.5,
    "methods":     ["sensitivity", "label_corr", "stability", "shap",
                    "mutual_info", "permutation_importance"],
    # ---- analysis ----
    "target":      "rev_drop",                   # also: "mean_drop", "worst_drop"
    "cluster_by":  "seed",                       # cluster bootstrap unit
    "n_boot":      2000,
    "alpha":       0.05,
}

RESULT_COLS = ["run_id", "model", "seed", "size", "corr", "reg", "aug",
               "method", "feature", "score", "spurious"]

def read_scores(path):
    # keep the literal feature name "null" (pandas would otherwise read it as NaN)
    return pd.read_csv(path, keep_default_na=False, na_values=[""])

def paths(cfg):
    d = cfg["out_dir"]
    return {
        "dir": d,
        "models":   f"{d}/models",
        "partial":  f"{d}/partial",
        "meta":     f"{d}/run_meta.csv",
        "frozen":   f"{d}/scores_frozen.csv",
        "manifest": f"{d}/scores_frozen.manifest.json",
        "ood_long": f"{d}/ood_drops_long.csv",
        "ood_wide": f"{d}/ood_drops.csv",
        "analysis": f"{d}/analysis_spearman.csv",
    }

def grid_runs(cfg):
    g = cfg["grid"]
    keys = ["arch", "train_corr", "size", "seed", "reg", "aug"]
    return [dict(zip(keys, v)) for v in itertools.product(*[g[k] for k in keys])]

def run_id_of(r):
    return f"{r['arch']}_c{r['train_corr']}_n{r['size']}_s{r['seed']}_r{r['reg']}_a{r['aug']}"

print(f"Output dir: {CONFIG['out_dir']}")
print(f"Grid size:  {len(grid_runs(CONFIG))} runs")

Output dir: /kaggle/working/runs_mi_perm_v2
Grid size:  40 runs


In [11]:
# MODEL FACTORY 
# All models take FLAT inputs (N, 3*28*28) because the shared score functions call
# model.predict(flat(X)). A CNN wrapper (Stage B) must reshape inside and have
# fit / predict / predict_proba, then be added to MODEL_REGISTRY.
MODEL_REGISTRY = {}

REG_LEVELS = {
    "lr":  {"low": 10.0, "default": 1.0,  "high": 0.01},   # C
    "rf":  {"low": None, "default": 12,   "high": 6},      # max_depth
    "mlp": {"low": 1e-5, "default": 1e-4, "high": 1e-2},   # L2 alpha
}

def make_model(arch, reg="default", seed=0):
    if arch in MODEL_REGISTRY:
        return MODEL_REGISTRY[arch](reg=reg, seed=seed)
    if arch == "lr":
        return LogisticRegression(C=REG_LEVELS["lr"][reg], max_iter=300)
    if arch == "rf":
        return RandomForestClassifier(n_estimators=200, max_depth=REG_LEVELS["rf"][reg],
                                      n_jobs=-1, random_state=seed)
    if arch == "mlp":
        return MLPClassifier(hidden_layer_sizes=(256,), alpha=REG_LEVELS["mlp"][reg],
                             max_iter=50, early_stopping=True, random_state=seed)
    raise ValueError(f"Unknown arch '{arch}'. Register it in MODEL_REGISTRY.")

In [12]:
# RUN DRIVER (pre-shift only: never touches test_x)
def _shap_cached(ctx, attr):
    if "_shap" not in ctx:
        ctx["_shap"] = attribute_shapley(ctx["model"], ctx["Xs"], ctx["ys"], ATTRS)
    return ctx["_shap"][attr]

# Every score method has the same shape: fn(ctx, attr) -> number.
# To add a new method (e.g. mutual information), just add it here.
SCORE_FNS = {
    "sensitivity": lambda ctx, a: sensitivity_score(ctx["model"], ctx["Xs"], a),
    "label_corr":  lambda ctx, a: label_corr_score(ctx["ytr"], ctx["Atr"], a),
    "stability":   lambda ctx, a: stability_score(ctx["model"], ctx["Xs"], ctx["ys"], ctx["As"], a),
    "shap":        _shap_cached,
    "mutual_info": lambda ctx, a: mutual_information_score(ctx["ytr"], ctx["Atr"], a),
    "permutation_importance": lambda ctx, a: grouped_attribute_permutation_score(
        ctx["model"], ctx["Xs"], ctx["ys"], ctx["As"][a], a,
        n_repeats=5, seed=12345),
}

def make_e1(cfg, train_corr, size, seed, aug):
    """Training environment E1 (+ held-out val). Uses train_x / val_x only."""
    corrs = {cfg["shortcut"]: train_corr, **cfg["decoy_corrs"]}
    rng = np.random.default_rng(10_000 + seed)
    idx = rng.choice(len(train_y), size=min(size, len(train_y)), replace=False)
    Xtr, Atr = make_multi_attr_mnist(train_x[idx], train_y[idx], corrs, seed=20_000 + seed)
    ytr = train_y[idx]

    if aug == "color_rand":                     # colour randomisation augmentation
        flip = np.random.default_rng(30_000 + seed).random(len(ytr)) < cfg["aug_p"]
        Xtr[flip] = intervene(Xtr[flip], "color")
        Atr["color"] = np.where(flip, 1 - Atr["color"], Atr["color"])
    elif aug != "none":
        raise ValueError(aug)

    Xva, Ava = make_multi_attr_mnist(val_x, val_y, corrs, seed=40_000 + seed)
    return Xtr, ytr, Atr, Xva, val_y, Ava

def run_one(cfg, r):
    P = paths(cfg)
    rid = run_id_of(r)
    Xtr, ytr, Atr, Xva, yva, Ava = make_e1(cfg, r["train_corr"], r["size"], r["seed"], r["aug"])
    design_corrs = {cfg["shortcut"]: r["train_corr"], **cfg["decoy_corrs"]}
    if r["aug"] == "color_rand":
        # Randomly swapping a fraction p attenuates designed correlation by (1 - 2p).
        design_corrs[cfg["shortcut"]] *= (1.0 - 2.0 * cfg["aug_p"])
    reg = build_registry(ytr, Atr, threshold=cfg["spurious_threshold"],
                         design_corrs=design_corrs)

    t0 = time.time()
    model = make_model(r["arch"], r["reg"], r["seed"]).fit(flat(Xtr), ytr)
    fit_s = time.time() - t0

    n = cfg["n_score"]
    ctx = {"model": model, "Xs": Xva[:n], "ys": yva[:n],
           "As": {a: v[:n] for a, v in Ava.items()}, "Atr": Atr, "ytr": ytr}

    base = {"run_id": rid, "model": r["arch"], "seed": r["seed"], "size": r["size"],
            "corr": r["train_corr"], "reg": r["reg"], "aug": r["aug"]}
    rows = []
    for m in cfg["methods"]:
        for a in ATTRS:
            rows.append({**base, "method": m, "feature": a,
                         "score": SCORE_FNS[m](ctx, a), "spurious": reg[a]["spurious"]})
    # null control: a do-nothing intervention should score 0
    rows.append({**base, "method": "sensitivity", "feature": "null",
                 "score": sensitivity_score(model, ctx["Xs"], "null"), "spurious": False})

    meta = {**base,
            "train_acc": accuracy_score(ytr, model.predict(flat(Xtr))),
            "id_val_acc": accuracy_score(yva, model.predict(flat(Xva))),
            "fit_seconds": round(fit_s, 2),
            "model_path": f"{P['models']}/{rid}.pkl"}
    for a in ATTRS:
        meta[f"traincorr_{a}"] = reg[a]["train_corr"]

    with open(meta["model_path"], "wb") as f:
        pickle.dump(model, f)
    pd.DataFrame(rows, columns=RESULT_COLS).to_csv(f"{P['partial']}/{rid}.csv", index=False)
    pd.DataFrame([meta]).to_csv(f"{P['partial']}/{rid}.meta.csv", index=False)
    return rows, meta

def run_grid(cfg, shard=(0, 1), verbose=True):
    """Runs every grid setting not already done. shard=(i, k) runs every k-th one starting at i,
    so a long grid can be split across several Kaggle versions."""
    P = paths(cfg)
    if os.path.exists(P["frozen"]):
        raise RuntimeError("Scores are already frozen; start a new out_dir for a new grid.")
    os.makedirs(P["models"], exist_ok=True); os.makedirs(P["partial"], exist_ok=True)
    runs = grid_runs(cfg)[shard[0]::shard[1]]
    for i, r in enumerate(runs):
        rid = run_id_of(r)
        if os.path.exists(f"{P['partial']}/{rid}.csv"):
            if verbose: print(f"[{i+1}/{len(runs)}] skip (done) {rid}")
            continue
        t0 = time.time()
        rows, meta = run_one(cfg, r)
        if verbose:
            s = {(x["method"], x["feature"]): x["score"] for x in rows}
            print(f"[{i+1}/{len(runs)}] {rid}  id_acc={meta['id_val_acc']:.3f}  "
                  f"sens(color)={s[('sensitivity','color')]:.3f}  "
                  f"sens(null)={s[('sensitivity','null')]:.3f}  ({time.time()-t0:.1f}s)")

In [13]:
# FREEZE WRITER
def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def freeze_scores(cfg):
    """Merge all score files into scores_frozen.csv. Refuses to overwrite."""
    P = paths(cfg)
    if os.path.exists(P["frozen"]):
        raise FileExistsError(f"{P['frozen']} is frozen and will not be overwritten.")
    expected = {run_id_of(r) for r in grid_runs(cfg)}
    done = {f[:-4] for f in os.listdir(P["partial"]) if f.endswith(".csv") and not f.endswith(".meta.csv")}
    missing = expected - done
    if missing:
        raise RuntimeError(f"{len(missing)} runs missing, e.g. {sorted(missing)[:3]}. Finish the grid first.")

    scores = pd.concat([read_scores(f"{P['partial']}/{r}.csv") for r in sorted(expected)], ignore_index=True)
    meta = pd.concat([pd.read_csv(f"{P['partial']}/{r}.meta.csv") for r in sorted(expected)], ignore_index=True)
    assert list(scores.columns) == RESULT_COLS, "results schema drifted"

    scores.to_csv(P["frozen"], index=False)
    meta.to_csv(P["meta"], index=False)
    manifest = {"sha256": _sha256(P["frozen"]), "meta_sha256": _sha256(P["meta"]),
                "n_rows": len(scores), "n_runs": len(expected),
                "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S"), "config": cfg}
    with open(P["manifest"], "w") as f:
        json.dump(manifest, f, indent=2, default=str)
    for p in (P["frozen"], P["meta"], P["manifest"]):
        os.chmod(p, 0o444)
    print(f"Frozen {len(scores)} score rows from {len(expected)} runs -> {P['frozen']}")
    print(f"sha256 = {manifest['sha256']}")
    return manifest

In [14]:
#  OOD EVALUATION
def verify_freeze(cfg):
    P = paths(cfg)
    if not (os.path.exists(P["frozen"]) and os.path.exists(P["manifest"])):
        raise RuntimeError("No frozen scores found. Run freeze_scores() before touching OOD data.")
    with open(P["manifest"]) as f:
        man = json.load(f)
    if _sha256(P["frozen"]) != man["sha256"] or _sha256(P["meta"]) != man["meta_sha256"]:
        raise RuntimeError("Frozen files were modified after freezing (hash mismatch).")
    return man

def ood_evaluate(cfg):
    P = paths(cfg)
    man = verify_freeze(cfg)
    meta = pd.read_csv(P["meta"])
    decoys = cfg["decoy_corrs"]

    # Same test images for every run, so drops are comparable
    env_cache = {}
    def env_data(c, seed):
        if c not in env_cache:
            X, _ = make_multi_attr_mnist(test_x, test_y, {cfg["shortcut"]: c, **decoys}, seed=seed)
            env_cache[c] = flat(X)
        return env_cache[c]

    rows = []
    for _, m in meta.iterrows():
        with open(m["model_path"], "rb") as f:
            model = pickle.load(f)
        # Baseline: test digits drawn with the run's own training correlation
        id_acc = accuracy_score(test_y, model.predict(env_data(m["corr"], 50_000)))
        for k, (env, c) in enumerate(cfg["ood_envs"].items()):
            acc = accuracy_score(test_y, model.predict(env_data(c, 60_000 + k)))
            rows.append({"run_id": m["run_id"], "model": m["model"], "seed": m["seed"],
                         "size": m["size"], "corr": m["corr"], "reg": m["reg"], "aug": m["aug"],
                         "env": env, "env_corr": c, "id_test_acc": id_acc,
                         "ood_acc": acc, "drop": id_acc - acc})
    long = pd.DataFrame(rows)

    keys = ["run_id", "model", "seed", "size", "corr", "reg", "aug", "id_test_acc"]
    wide = long.pivot_table(index=keys, columns="env", values="drop").add_suffix("_drop").reset_index()
    wide.columns.name = None
    std_envs = [f"{e}_drop" for e in cfg["ood_envs"] if e != "REV"]
    wide["mean_drop"] = wide[std_envs].mean(axis=1)
    wide["worst_drop"] = wide[[f"{e}_drop" for e in cfg["ood_envs"]]].max(axis=1)
    wide["rev_drop"] = wide["REV_drop"] if "REV_drop" in wide else np.nan
    wide["frozen_sha256"] = man["sha256"]

    long.to_csv(P["ood_long"], index=False)
    wide.to_csv(P["ood_wide"], index=False)
    print(f"OOD evaluation done for {len(meta)} runs -> {P['ood_wide']}")
    return long, wide

In [15]:
# ANALYSIS 
def _partial_spearman(x, y, z):
    """Spearman of x and y after removing the effect of z."""
    rx, ry, rz = (pd.Series(v).rank().to_numpy() for v in (x, y, z))
    Z = np.column_stack([np.ones_like(rz), rz])
    ex = rx - Z @ np.linalg.lstsq(Z, rx, rcond=None)[0]
    ey = ry - Z @ np.linalg.lstsq(Z, ry, rcond=None)[0]
    if ex.std() < 1e-12 or ey.std() < 1e-12:
        return np.nan, np.nan
    r = float(np.corrcoef(ex, ey)[0, 1])
    n = len(x); df = n - 3
    if df <= 0 or abs(r) >= 1:
        return r, np.nan
    tstat = r * np.sqrt(df / (1 - r * r))
    return r, float(2 * t_dist.sf(abs(tstat), df))

def _cluster_bootstrap_ci(df, xcol, ycol, cluster, n_boot, alpha, seed=0):
    rng = np.random.default_rng(seed)
    groups = {g: d for g, d in df.groupby(cluster)}
    keys = list(groups)
    stats = []
    for _ in range(n_boot):
        pick = rng.choice(keys, size=len(keys), replace=True)
        b = pd.concat([groups[k] for k in pick])
        if b[xcol].nunique() > 1 and b[ycol].nunique() > 1:
            stats.append(spearmanr(b[xcol], b[ycol])[0])
    if len(stats) < 10:
        return np.nan, np.nan
    return tuple(np.nanpercentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)]))

def _holm(pvals):
    p = np.asarray(pvals, float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    idx = np.argsort(p[ok]); m = ok.sum()
    adj = np.maximum.accumulate((m - np.arange(m)) * p[ok][idx]).clip(max=1)
    tmp = np.empty(m); tmp[idx] = adj
    out[ok] = tmp
    return out

def load_analysis_table(cfg, feature=None):
    P = paths(cfg)
    verify_freeze(cfg)
    scores = read_scores(P["frozen"])
    ood = pd.read_csv(P["ood_wide"])
    feature = feature or cfg["shortcut"]
    wide = (scores[scores.feature == feature]
            .pivot_table(index="run_id", columns="method", values="score")
            .reset_index())
    wide.columns.name = None
    return scores, wide.merge(ood, on="run_id")

def analyze(cfg, target=None, feature=None, plot=True):
    target = target or cfg["target"]
    scores, df = load_analysis_table(cfg, feature)
    methods = [m for m in cfg["methods"] if m in df]

    # 1) Does each frozen score predict the later OOD drop?
    res = []
    for m in methods:
        d = df[list(dict.fromkeys([m, target, "label_corr", cfg["cluster_by"], "model"]))].dropna()
        if len(d) < 3 or d[m].nunique() < 2:
            res.append({"method": m, "n": len(d)}); continue
        rho, p = spearmanr(d[m], d[target])
        lo, hi = _cluster_bootstrap_ci(d, m, target, cfg["cluster_by"], cfg["n_boot"], cfg["alpha"])
        if m != "label_corr":
            prho, pp = _partial_spearman(d[m].values, d[target].values, d["label_corr"].values)
        else:
            prho, pp = np.nan, np.nan
        res.append({"method": m, "n": len(d), "spearman": rho, "ci_lo": lo, "ci_hi": hi,
                    "p": p, "partial_rho|label_corr": prho, "partial_p": pp})
    res = pd.DataFrame(res)
    if "p" in res:
        res["p_holm"] = _holm(res["p"].values)
        res["partial_p_holm"] = _holm(res["partial_p"].values)

    # 2) Grouped analysis (mixed-effects model)
    mixed = None
    try:
        import statsmodels.formula.api as smf
        warnings.filterwarnings("ignore", module="statsmodels")
        grp = "model" if df["model"].nunique() > 1 else cfg["cluster_by"]
        d = df.dropna(subset=["sensitivity", "label_corr", target]).copy()
        for c in ["sensitivity", "label_corr"]:
            d[c + "_z"] = (d[c] - d[c].mean()) / (d[c].std() + 1e-12)
        fit = smf.mixedlm(f"{target} ~ sensitivity_z + label_corr_z", d, groups=d[grp]).fit(reml=False)
        mixed = fit.summary().tables[1]
    except Exception as e:
        print(f"(mixed-effects model skipped: {type(e).__name__}: {e})")

    # 3) Null / decoy controls: colour should clearly beat both
    sens = (scores[scores.method == "sensitivity"]
            .pivot_table(index=["run_id", "corr"], columns="feature", values="score").reset_index())
    decoy_cols = [a for a in ATTRS if a != cfg["shortcut"]]
    sens["shortcut_beats_controls"] = sens[cfg["shortcut"]] > sens[decoy_cols + ["null"]].max(axis=1)
    controls = (sens.groupby("corr")[[cfg["shortcut"], *decoy_cols, "null", "shortcut_beats_controls"]]
                .mean().round(4))

    # 4) Blind detection on the grid (reuses Person 3's evaluate())
    bd = (scores[scores.feature.isin(ATTRS)]
          .pivot_table(index=["run_id", "feature"], columns="method", values="score").reset_index()
          .merge(scores[["run_id", "feature", "spurious"]].drop_duplicates(), on=["run_id", "feature"])
          .rename(columns={"run_id": "task", "feature": "attr"}))
    strongest = {t: cfg["shortcut"] for t in bd.task.unique()
                 if bd[(bd.task == t) & (bd.attr == cfg["shortcut"])].spurious.iloc[0]}
    _, blind = evaluate(bd[bd.task.isin(strongest)], strongest, methods=tuple(methods))

    print(f"\n=== Score vs {target} (feature = {feature or cfg['shortcut']}) ===")
    print(res.round(4).to_string(index=False))
    if mixed is not None:
        print("\n=== Mixed-effects model ===\n", mixed)
    print("\n=== Sensitivity controls (mean per training corr) ===\n", controls.to_string())
    print("\n=== Blind detection on grid runs ===\n", blind.round(4).to_string())

    P = paths(cfg)
    res.to_csv(P["analysis"], index=False)

    if plot:
        fig, axes = plt.subplots(1, len(methods), figsize=(4.2 * len(methods), 3.8), squeeze=False)
        for ax, m in zip(axes[0], methods):
            sc = ax.scatter(df[m], df[target], c=df["corr"], cmap="viridis", s=28)
            r = res.set_index("method").loc[m]
            ax.set_title(f"{m}\nρ={r.get('spearman', np.nan):.2f}")
            ax.set_xlabel(f"frozen {m} score"); ax.set_ylabel(target)
            ax.grid(alpha=0.3)
        fig.colorbar(sc, ax=axes[0].tolist(), label="training corr")
        plt.show()
    return res, controls, blind, mixed

In [16]:
# SMOKE TEST 
def smoke_test():
    cfg = copy.deepcopy(CONFIG)
    cfg["out_dir"] = f"{WORK}/runs_smoke"
    cfg["grid"] = {"arch": ["lr"], "train_corr": [0.6, 0.95], "size": [1500],
                   "seed": [0, 1], "reg": ["default"], "aug": ["none"]}
    cfg["n_score"] = 300
    cfg["n_boot"] = 200
    if os.path.exists(cfg["out_dir"]):
        for root, _, files in os.walk(cfg["out_dir"]):
            for f in files: os.chmod(os.path.join(root, f), 0o644)
        shutil.rmtree(cfg["out_dir"])
    P = paths(cfg)

    # The test-time check must be blocked before freezing
    try:
        os.makedirs(P["dir"], exist_ok=True); ood_evaluate(cfg)
        raise AssertionError("OOD eval ran before freeze!")
    except RuntimeError:
        pass

    run_grid(cfg)
    freeze_scores(cfg)
    try:
        freeze_scores(cfg); raise AssertionError("freeze overwrote frozen scores!")
    except FileExistsError:
        pass

    scores = read_scores(P["frozen"])
    assert list(scores.columns) == RESULT_COLS
    assert scores.run_id.nunique() == len(grid_runs(cfg))
    s = scores[scores.method == "sensitivity"].set_index(["run_id", "feature"])["score"]
    hi = [r for r in scores.run_id.unique() if "_c0.95_" in r]
    for r in scores.run_id.unique():
        assert s[(r, "null")] == 0.0, "null intervention must not change predictions"
    for r in hi:
        assert s[(r, "color")] > max(s[(r, "marker_tl")], s[(r, "marker_br")]), "colour should beat decoys"

    long, wide = ood_evaluate(cfg)
    assert {"mean_drop", "rev_drop", "worst_drop"} <= set(wide.columns)
    assert (wide[wide["corr"] == 0.95]["rev_drop"] > 0).all(), "reversed colour should hurt a 0.95 model"

    assert set(scores.method.unique()) == set(cfg["methods"]), "score table must contain all six methods"
    pi = scores[(scores.method == "permutation_importance") & (scores.feature == "color")].set_index("run_id")["score"]
    for r in hi:
        assert pi[r] > 0, "shuffling colour should hurt a high-correlation model"

    res, controls, blind, _ = analyze(cfg, plot=False)
    assert len(res) == len(cfg["methods"])
    print("\n SMOKE TEST PASSED")
    return cfg

# (LR smoke test definition kept unchanged; the CNN smoke test is below)

## Part 2 — CNN model (writes `cnn_model.py` into /kaggle/working so the merge notebook can load the pickled models)

In [17]:
%%writefile /kaggle/working/cnn_model.py
"""
cnn_model.py — Stage B, Person 4: small CNN for the Spurious-Feature project.

Plugs into the frozen shared core WITHOUT changing any score function:
  * takes FLAT inputs (N, 3*28*28) like every other model, reshapes inside
  * exposes fit / predict / predict_proba / classes_ (sklearn-style)
  * pickles to CPU tensors, so the OOD/merge notebook can load it with or
    without a GPU (the merge notebook must be able to `import cnn_model`)
  * deterministic given `seed` (same seed -> same weights on the same device)
  * early stopping uses an internal split of the TRAINING data only (E1),
    never the scoring/validation set and never any OOD data

Usage in the CNN notebook (after `from core import *`):
    import cnn_model
    cnn_model.register(MODEL_REGISTRY, REG_LEVELS)
"""
import os
import copy
import numpy as np

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # needed for deterministic cuBLAS
import torch
import torch.nn as nn
import torch.nn.functional as F

IMG_SHAPE = (3, 28, 28)

# Regularisation levels = Adam L2 weight decay (parallels the MLP's `alpha`).
CNN_REG_LEVELS = {"low": 0.0, "default": 1e-4, "high": 1e-2}

# Hyperparameters. Fix these on E1 validation ONLY, before anyone looks at OOD numbers,
# then do not change them (they are written into the frozen manifest via CONFIG).
CNN_HPARAMS = {
    "channels": (16, 32),     # conv widths
    "hidden": 64,             # fully connected width
    "lr": 3e-3,               # chosen on E1 validation (tuning_e1_only.csv)
    "batch_size": 128,
    "max_epochs": 15,
    "patience": 3,            # early stopping on the internal E1 split
    "inner_val_frac": 0.1,
}


def _device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


class SmallCNN(nn.Module):
    """2 conv blocks + 1 hidden FC layer, ~110k parameters.
    'same' padding keeps the 3x3 corner markers visible to the first conv layer."""

    def __init__(self, channels=(16, 32), hidden=64, n_classes=2):
        super().__init__()
        c1, c2 = channels
        self.conv1 = nn.Conv2d(3, c1, 3, padding=1)
        self.conv2 = nn.Conv2d(c1, c2, 3, padding=1)
        self.fc1 = nn.Linear(c2 * 7 * 7, hidden)
        self.fc2 = nn.Linear(hidden, n_classes)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)   # 28 -> 14
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)   # 14 -> 7
        x = F.relu(self.fc1(x.flatten(1)))
        return self.fc2(x)


class CNNClassifier:
    """sklearn-style wrapper so the shared core can call model.predict(flat(X))."""

    def __init__(self, weight_decay=1e-4, seed=0, **hparams):
        self.weight_decay = weight_decay
        self.seed = seed
        self.hp = {**CNN_HPARAMS, **hparams}
        self.classes_ = np.array([0, 1])
        self.net = None
        self.history_ = []          # per-epoch (train_loss, inner_val_acc)
        self.best_epoch_ = None

    # ---------- helpers ----------
    @staticmethod
    def _to_img(X):
        X = np.asarray(X, dtype=np.float32)
        return X.reshape(len(X), *IMG_SHAPE)

    def _seed_everything(self):
        torch.manual_seed(self.seed)
        np.random.seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)

    # ---------- sklearn API ----------
    def fit(self, X, y):
        self._seed_everything()
        dev = _device()
        X = self._to_img(X)
        y = np.asarray(y, dtype=np.int64)

        # internal split of the E1 TRAINING data for early stopping
        rng = np.random.default_rng(self.seed)
        perm = rng.permutation(len(y))
        n_in = max(1, int(self.hp["inner_val_frac"] * len(y)))
        iv, tr = perm[:n_in], perm[n_in:]

        Xtr = torch.from_numpy(X[tr]).to(dev); ytr = torch.from_numpy(y[tr]).to(dev)
        Xiv = torch.from_numpy(X[iv]).to(dev); yiv = torch.from_numpy(y[iv]).to(dev)

        self.net = SmallCNN(self.hp["channels"], self.hp["hidden"]).to(dev)
        opt = torch.optim.Adam(self.net.parameters(), lr=self.hp["lr"],
                               weight_decay=self.weight_decay)
        g = torch.Generator(device="cpu").manual_seed(self.seed)
        bs = self.hp["batch_size"]

        best_acc, best_state, bad = -1.0, None, 0
        self.history_ = []
        for epoch in range(self.hp["max_epochs"]):
            self.net.train()
            order = torch.randperm(len(ytr), generator=g).to(dev)
            tot = 0.0
            for i in range(0, len(order), bs):
                b = order[i:i + bs]
                opt.zero_grad(set_to_none=True)
                loss = F.cross_entropy(self.net(Xtr[b]), ytr[b])
                loss.backward()
                opt.step()
                tot += loss.item() * len(b)
            acc = float((self._logits(Xiv).argmax(1) == yiv).float().mean())
            self.history_.append((tot / len(ytr), acc))
            if acc > best_acc + 1e-4:
                best_acc, bad, self.best_epoch_ = acc, 0, epoch + 1
                best_state = copy.deepcopy(self.net.state_dict())
            else:
                bad += 1
                if bad >= self.hp["patience"]:
                    break
        self.net.load_state_dict(best_state)
        self.net.eval()
        return self

    @torch.no_grad()
    def _logits(self, Ximg, bs=2048):
        self.net.eval()
        out = [self.net(Ximg[i:i + bs]) for i in range(0, len(Ximg), bs)]
        return torch.cat(out)

    @torch.no_grad()
    def predict_proba(self, X):
        dev = next(self.net.parameters()).device
        Xt = torch.from_numpy(self._to_img(X)).to(dev)
        return F.softmax(self._logits(Xt), dim=1).cpu().numpy().astype(np.float64)

    def predict(self, X):
        return self.predict_proba(X).argmax(1)

    # ---------- pickling: always store CPU weights ----------
    def __getstate__(self):
        d = self.__dict__.copy()
        if self.net is not None:
            d["net"] = None
            d["_state_cpu"] = {k: v.detach().cpu() for k, v in self.net.state_dict().items()}
        return d

    def __setstate__(self, d):
        state = d.pop("_state_cpu", None)
        self.__dict__.update(d)
        if state is not None:
            self.net = SmallCNN(self.hp["channels"], self.hp["hidden"])
            self.net.load_state_dict(state)
            self.net.to(_device()).eval()

    def n_params(self):
        return sum(p.numel() for p in self.net.parameters())


def register(model_registry, reg_levels=None):
    """Add 'cnn' to the shared core's MODEL_REGISTRY (make_model checks it first)."""
    if reg_levels is not None:
        reg_levels["cnn"] = dict(CNN_REG_LEVELS)
    model_registry["cnn"] = lambda reg="default", seed=0: CNNClassifier(
        weight_decay=CNN_REG_LEVELS[reg], seed=seed)
    return model_registry


# =====================================================================
# Optional CNN-only diagnostics (exploratory; NOT pooled with other models)
# =====================================================================
def integrated_gradients_regions(model, X, y, patch, steps=32, baseline=None):
    """Integrated Gradients (black-image baseline unless given), aggregated into regions:
    red channel, green channel, each blue marker patch, rest of blue channel.
    Returns the mean share of |attribution| per region.

    Caveat for the write-up: colour and shape live in the SAME pixels (the stroke is drawn in
    the red OR green channel), so pixel attributions cannot separate 'uses colour' from
    'uses shape'. That is why the attribute-level counterfactual Shapley in the shared core is
    the primary SHAP-style score; this is only a sanity view."""
    net = model.net
    dev = next(net.parameters()).device
    x = torch.from_numpy(model._to_img(X)).to(dev)
    b = torch.zeros_like(x) if baseline is None else torch.from_numpy(model._to_img(baseline)).to(dev)
    yt = torch.from_numpy(np.asarray(y, dtype=np.int64)).to(dev)
    total = torch.zeros_like(x)
    for a in torch.linspace(0, 1, steps, device=dev):
        xi = (b + a * (x - b)).requires_grad_(True)
        out = net(xi).gather(1, yt[:, None]).sum()
        total += torch.autograd.grad(out, xi)[0]
    attr = ((x - b) * total / steps).abs().detach().cpu().numpy()
    mass = attr.reshape(len(attr), -1).sum(1) + 1e-12
    blue = attr[:, 2].copy()
    shares = {"red": attr[:, 0].reshape(len(attr), -1).sum(1) / mass,
              "green": attr[:, 1].reshape(len(attr), -1).sum(1) / mass}
    for name, (r, c) in patch.items():
        shares[name] = blue[:, r, c].reshape(len(blue), -1).sum(1) / mass
        blue[:, r, c] = 0
    shares["blue_other"] = blue.reshape(len(blue), -1).sum(1) / mass
    return {k: float(v.mean()) for k, v in shares.items()}


def realism_auroc(X_cf, X_nat, seed=0, epochs=5):
    """Realism check from the plan: can a CNN tell counterfactual images from NATURAL images
    with the same label and attribute value? Train on half, report AUROC on the other half.
    AUROC ~ 0.5 => counterfactuals are indistinguishable from real data."""
    from sklearn.metrics import roc_auc_score
    n = min(len(X_cf), len(X_nat))
    rng = np.random.default_rng(seed)
    X = np.concatenate([X_cf[rng.permutation(len(X_cf))[:n]],
                        X_nat[rng.permutation(len(X_nat))[:n]]])
    y = np.r_[np.ones(n, int), np.zeros(n, int)]
    p = rng.permutation(len(y)); X, y = X[p], y[p]
    h = len(y) // 2
    clf = CNNClassifier(seed=seed, max_epochs=epochs, patience=epochs)
    clf.fit(X[:h].reshape(h, -1), y[:h])
    s = clf.predict_proba(X[h:].reshape(len(y) - h, -1))[:, 1]
    return float(roc_auc_score(y[h:], s)), n


Writing /kaggle/working/cnn_model.py


In [18]:
import sys, torch
sys.path.insert(0, "/kaggle/working")
import cnn_model
cnn_model.register(MODEL_REGISTRY, REG_LEVELS)
print("Device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("CNN hyperparameters:", cnn_model.CNN_HPARAMS)
print("CNN reg levels (weight decay):", cnn_model.CNN_REG_LEVELS)

Device: cuda - Tesla T4
CNN hyperparameters: {'channels': (16, 32), 'hidden': 64, 'lr': 0.003, 'batch_size': 128, 'max_epochs': 15, 'patience': 3, 'inner_val_frac': 0.1}
CNN reg levels (weight decay): {'low': 0.0, 'default': 0.0001, 'high': 0.01}


In [19]:
# PART 6 (v2) - Blind detection with all SIX scores (MI + permutation importance added)
# Replaces Part 6. Uses the updated run_task/evaluate from the latest Notebook 0.
# Needs Parts 1-2 run first (data, shared core, cnn_model). Does NOT touch the frozen grid.

# ---- updated run_task / evaluate, copied verbatim from the latest Notebook 0 ----
def run_task(task_id, model_fn, n_score=2000):
    rng = np.random.default_rng(1000 + task_id)
    order = rng.permutation(ATTRS)
    corrs = {order[0]: 0.9, order[1]: 0.6, order[2]: 0.0}   # strong, weak, decoy

    Xtr, Atr = make_multi_attr_mnist(train_x, train_y, corrs, seed=2000 + task_id)
    Xva, Ava = make_multi_attr_mnist(val_x,   val_y,   corrs, seed=3000 + task_id)
    reg = build_registry(train_y, Atr, threshold=0.5, design_corrs=corrs)

    model = model_fn().fit(flat(Xtr), train_y)
    Xs, ys = Xva[:n_score], val_y[:n_score]
    As = {a: v[:n_score] for a, v in Ava.items()}
    shap = attribute_shapley(model, Xs, ys, ATTRS)

    rows = []
    for a in ATTRS:
        rows.append({
            "task": task_id, "attr": a, "true_corr": corrs[a],
            "spurious": reg[a]["spurious"],
            "sensitivity": sensitivity_score(model, Xs, a),
            "label_corr":  label_corr_score(train_y, Atr, a),
            "stability":   stability_score(model, Xs, ys, As, a),
            "shap":        shap[a],
            "mutual_info": mutual_information_score(train_y, Atr, a),
            "permutation_importance": grouped_attribute_permutation_score(
                model, Xs, ys, As[a], a, n_repeats=5, seed=12345),
        })
    null = sensitivity_score(model, Xs, "null")
    return rows, order[0], null

def evaluate(all_rows, strongest,
             methods=("sensitivity", "label_corr", "stability", "shap",
                      "mutual_info", "permutation_importance")):
    df = pd.DataFrame(all_rows)
    out = {}
    for m in methods:
        top1 = np.mean([df[df.task == t].sort_values(m, ascending=False).attr.iloc[0] == s
                        for t, s in strongest.items()])
        sub = df.dropna(subset=[m])
        auroc = roc_auc_score(sub.spurious, sub[m]) if sub.spurious.nunique() == 2 else np.nan
        out[m] = {"top1": top1, "auroc": auroc}
    return df, pd.DataFrame(out).T

# ---- rerun the 10 blind-detection tasks with the CNN ----
cnn_fn_for = lambda t: (lambda: make_model("cnn", "default", seed=t))
rows_c, strongest_c, nulls_c = [], {}, []
for t in range(10):
    r, s, null = run_task(t, cnn_fn_for(t))
    rows_c += r; strongest_c[t] = s; nulls_c.append(null)
    print(f"task {t}: shortcut = {s}, null sensitivity = {null:.3f}")
df_c, summary_c = evaluate(rows_c, strongest_c)
print(summary_c)
assert set(summary_c.index) == {"sensitivity", "label_corr", "stability", "shap",
                                "mutual_info", "permutation_importance"}, "expected all six scores"
df_c.to_csv(f"{WORK}/blind_detection_cnn_v2.csv", index=False)
print("Saved", f"{WORK}/blind_detection_cnn_v2.csv")

task 0: shortcut = color, null sensitivity = 0.000
task 1: shortcut = marker_br, null sensitivity = 0.000
task 2: shortcut = color, null sensitivity = 0.000
task 3: shortcut = marker_br, null sensitivity = 0.000
task 4: shortcut = color, null sensitivity = 0.000
task 5: shortcut = color, null sensitivity = 0.000
task 6: shortcut = marker_br, null sensitivity = 0.000
task 7: shortcut = marker_br, null sensitivity = 0.000
task 8: shortcut = marker_tl, null sensitivity = 0.000
task 9: shortcut = marker_br, null sensitivity = 0.000
                        top1  auroc
sensitivity              1.0  1.000
label_corr               1.0  1.000
stability                1.0  0.985
shap                     1.0  1.000
mutual_info              1.0  1.000
permutation_importance   1.0  1.000
Saved /kaggle/working/blind_detection_cnn_v2.csv


## Part 3 — CNN config
Same grid, seeds, decoys and OOD environments as the LR run, so CNN and LR runs are **paired** on identical E1 data. Only `arch` and `out_dir` change.

In [ ]:
CONFIG_CNN = copy.deepcopy(CONFIG)
CONFIG_CNN["out_dir"] = f"{WORK}/runs_cnn"
CONFIG_CNN["grid"]["arch"] = ["cnn"]
# Stretch (only if GPU time allows; the MLP owner runs the main reg/aug sweeps):
# CONFIG_CNN["grid"]["reg"] = ["low", "default", "high"]
# CONFIG_CNN["grid"]["aug"] = ["none", "color_rand"]
CONFIG_CNN["cnn_hparams"] = dict(cnn_model.CNN_HPARAMS)        # recorded in the frozen manifest
CONFIG_CNN["cnn_reg_levels"] = dict(cnn_model.CNN_REG_LEVELS)
print(f"{len(grid_runs(CONFIG_CNN))} CNN runs -> {CONFIG_CNN['out_dir']}")

## Part 4 — CNN smoke test (same checks as the Stage A smoke test, with arch = cnn)

In [ ]:

def smoke_test_cnn():
    cfg = copy.deepcopy(CONFIG)
    cfg["out_dir"] = f"{WORK}/runs_smoke_cnn"
    cfg["grid"] = {"arch": ["cnn"], "train_corr": [0.6, 0.95], "size": [1500],
                   "seed": [0, 1], "reg": ["default"], "aug": ["none"]}
    cfg["n_score"] = 300; cfg["n_boot"] = 200
    if os.path.exists(cfg["out_dir"]):
        for root, _, files in os.walk(cfg["out_dir"]):
            for f in files: os.chmod(os.path.join(root, f), 0o644)
        shutil.rmtree(cfg["out_dir"])
    P = paths(cfg)
    try:
        os.makedirs(P["dir"], exist_ok=True); ood_evaluate(cfg); raise AssertionError("OOD ran before freeze")
    except RuntimeError: pass
    run_grid(cfg); freeze_scores(cfg)
    try: freeze_scores(cfg); raise AssertionError("freeze overwrote")
    except FileExistsError: pass
    scores = read_scores(P["frozen"])
    assert list(scores.columns) == RESULT_COLS
    s = scores[scores.method == "sensitivity"].set_index(["run_id", "feature"])["score"]
    for r in scores.run_id.unique():
        assert s[(r, "null")] == 0.0
        if "_c0.95_" in r: assert s[(r, "color")] > max(s[(r, "marker_tl")], s[(r, "marker_br")])
    assert set(scores.method.unique()) == set(cfg["methods"]), "score table must contain all six methods"
    pi = scores[(scores.method == "permutation_importance") & (scores.feature == "color")].set_index("run_id")["score"]
    for r in [x for x in scores.run_id.unique() if "_c0.95_" in x]:
        assert pi[r] > 0, "shuffling colour should hurt a high-correlation model"
    long, wide = ood_evaluate(cfg)
    assert (wide[wide["corr"] == 0.95]["rev_drop"] > 0).all()
    analyze(cfg, plot=False)
    print("\n CNN SMOKE TEST PASSED")
smoke_test_cnn()


## Part 5 — Main CNN grid → freeze (pre-shift only)
`run_grid` skips runs already done, so if a session dies you can re-run. To split across Kaggle versions use `shard=(0,2)` then `shard=(1,2)`.

In [ ]:
run_grid(CONFIG_CNN)
_ = freeze_scores(CONFIG_CNN)

## Part 6 — Blind detection with the CNN (10 generated tasks, shortcut identity varies)
Same `run_task` / `evaluate` as Person 3's LR version.

In [ ]:
cnn_fn_for = lambda t: (lambda: make_model("cnn", "default", seed=t))
rows_c, strongest_c, nulls_c = [], {}, []
for t in range(10):
    r, s, null = run_task(t, cnn_fn_for(t))
    rows_c += r; strongest_c[t] = s; nulls_c.append(null)
    print(f"task {t}: shortcut = {s}, null sensitivity = {null:.3f}")
df_c, summary_c = evaluate(rows_c, strongest_c)
print(summary_c)
df_c.to_csv(f"{WORK}/blind_detection_cnn.csv", index=False)

## Part 7 — Realism check (validity section)
Can a CNN tell colour-swapped counterfactuals from natural images with the **same label and colour**? AUROC ≈ 0.5 means no. A positive control with a deliberate faint artifact must score ≈ 1.0, otherwise the check has no power.

In [ ]:
from cnn_model import realism_auroc
realism_rows = []
for corr in CONFIG_CNN["grid"]["train_corr"]:
    Xtr, ytr, Atr, Xva, yva, Ava = make_e1(CONFIG_CNN, corr, 5000, 0, "none")
    Xn, An = make_multi_attr_mnist(val_x, val_y, {"color": 0.0, "marker_tl": 0.0, "marker_br": 0.0}, seed=777)
    half = len(yva) // 2
    Xcf = intervene(Xva[:half], "color"); c_after = 1 - Ava["color"][:half]; y_cf = yva[:half]
    cf, nat = [], []
    for y in (0, 1):
        for c in (0, 1):
            a = Xcf[(y_cf == y) & (c_after == c)]
            b = Xn[half:][(val_y[half:] == y) & (An["color"][half:] == c)]
            k = min(len(a), len(b)); cf.append(a[:k]); nat.append(b[:k])
    cf, nat = np.concatenate(cf), np.concatenate(nat)
    au, n = realism_auroc(cf, nat, seed=0)
    bad = cf.copy(); bad[:, 2] += 0.03                      # positive control: faint blue haze
    au_bad, _ = realism_auroc(bad, nat, seed=0)
    realism_rows.append({"train_corr": corr, "n_per_class": n, "realism_auroc_color_swap": au,
                         "positive_control_auroc": au_bad})
    print(realism_rows[-1])
pd.DataFrame(realism_rows).to_csv(f"{WORK}/realism_check_cnn.csv", index=False)

## Part 8 — OOD evaluation + analysis (runs only after the freeze; the core refuses otherwise)

In [ ]:
ood_long_c, ood_wide_c = ood_evaluate(CONFIG_CNN)
results_c, controls_c, blind_c, mixed_c = analyze(CONFIG_CNN, target="rev_drop")
_ = analyze(CONFIG_CNN, target="mean_drop", plot=False)

## Part 9 — CNN-specific views (exploratory, not pooled)
(a) Colour reliance vs training correlation and dataset size. (b) Integrated-gradients region shares — note that colour and shape share the same pixels, so this **cannot** isolate colour use; it is a sanity view only.

In [ ]:
sc = read_scores(paths(CONFIG_CNN)["frozen"])
meta_c = pd.read_csv(paths(CONFIG_CNN)["meta"])
tab = (sc[(sc.method == "sensitivity") & (sc.feature == "color")]
       .groupby(["corr", "size"]).score.agg(["mean", "std"]).round(3))
print(tab)
print(meta_c.groupby(["corr", "size"])[["id_val_acc", "fit_seconds"]].mean().round(3))

from cnn_model import integrated_gradients_regions
ig_rows = []
for _, m in meta_c[meta_c.seed == 0].iterrows():
    with open(m["model_path"], "rb") as f: mdl = pickle.load(f)
    _, _, _, Xva, yva, _ = make_e1(CONFIG_CNN, m["corr"], m["size"], m["seed"], m["aug"])
    ig_rows.append({"run_id": m["run_id"], **integrated_gradients_regions(mdl, Xva[:300], yva[:300], PATCH)})
pd.DataFrame(ig_rows).to_csv(f"{WORK}/ig_regions_cnn.csv", index=False)
print(pd.DataFrame(ig_rows).round(3).to_string(index=False))

## Part 10 — Hand-off
Give the merge notebook: `runs_cnn/` (scores_frozen.csv + manifest, run_meta.csv, ood_drops.csv, models/), `cnn_model.py`, `blind_detection_cnn.csv`, `realism_check_cnn.csv`, `ig_regions_cnn.csv`. The merge notebook must run `sys.path.insert(0, <path to this notebook's output>)` before unpickling CNN models.